# Moustaghfir Ahmed Yassine - Prompt engineering pour l'analyse de sentiments

Ce notebook reprend la partie de la vidéo `wckvmfDESP8` : construire et évaluer des prompts pour classer des avis IMDb en `positive` ou `negative`.

In [ ]:
import json
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from tqdm.auto import tqdm
from IPython.display import display

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [ ]:
def charger_imdb(max_rows=2000):
    """Charge IMDb depuis Hugging Face. Si le réseau est indisponible, utilise un mini-jeu local."""
    try:
        dataset = load_dataset("imdb", split="train")
        df = dataset.to_pandas()[["text", "label"]]
        df = df.sample(n=min(max_rows, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
        print("Dataset IMDb charge depuis Hugging Face.")
    except Exception as exc:
        print("Fallback local utilise, raison :", exc)
        df = pd.DataFrame(
            [
                {"text": "This movie was wonderful, emotional and beautifully acted.", "label": 1},
                {"text": "A boring film with weak characters and a terrible ending.", "label": 0},
                {"text": "I loved the story and the soundtrack was amazing.", "label": 1},
                {"text": "The plot was predictable and the acting was awful.", "label": 0},
                {"text": "A brilliant and inspiring movie that I would watch again.", "label": 1},
                {"text": "I regret watching it; it was slow and confusing.", "label": 0},
            ]
        )
    return df

train_df = charger_imdb()
train_df["sentiment"] = np.where(train_df["label"] == 1, "positive", "negative")
train_df.head()

In [ ]:
display(train_df["sentiment"].value_counts())
train_df["sentiment"].value_counts().plot(kind="bar", title="Distribution des sentiments")
plt.xlabel("Sentiment")
plt.ylabel("Nombre d'avis")
plt.show()

examples_df, gold_examples_df = train_test_split(
    train_df[["text", "sentiment"]],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train_df["sentiment"] if train_df["sentiment"].nunique() == 2 else None,
)

print("Examples :", examples_df.shape)
print("Gold examples :", gold_examples_df.shape)

gold_examples = (
    gold_examples_df.sample(n=min(20, len(gold_examples_df)), random_state=34)
    .to_json(orient="records", force_ascii=False)
)

json.loads(gold_examples)[:2]

In [ ]:
USER_MESSAGE_TEMPLATE = """```{movie_review}```"""

ZERO_SHOT_SYSTEM_MESSAGE = """
Classify the sentiment of the movie review as exactly one label: positive or negative.
Do not explain your answer.
""".strip()

FEW_SHOT_SYSTEM_MESSAGE = ZERO_SHOT_SYSTEM_MESSAGE

COT_FEW_SHOT_SYSTEM_MESSAGE = """
Classify the sentiment of the movie review as exactly one label: positive or negative.
Read the review carefully before answering.
Consider only the content of the review.
Return only positive or negative, without explanation.
""".strip()


def create_examples(dataset, n=2, random_state=None):
    positive = dataset[dataset["sentiment"] == "positive"][["text", "sentiment"]]
    negative = dataset[dataset["sentiment"] == "negative"][["text", "sentiment"]]

    n_pos = min(n, len(positive))
    n_neg = min(n, len(negative))

    selected = pd.concat(
        [
            positive.sample(n=n_pos, random_state=random_state),
            negative.sample(n=n_neg, random_state=random_state),
        ]
    )
    selected = selected.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return selected.to_json(orient="records", force_ascii=False)


def create_prompt(system_message, examples_json=None, user_message_template=USER_MESSAGE_TEMPLATE):
    messages = [{"role": "system", "content": system_message}]

    if examples_json:
        for example in json.loads(examples_json):
            messages.append(
                {
                    "role": "user",
                    "content": user_message_template.format(movie_review=example["text"]),
                }
            )
            messages.append({"role": "assistant", "content": example["sentiment"]})

    return messages


few_shot_examples = create_examples(examples_df, n=2, random_state=RANDOM_STATE)
zero_shot_prompt = create_prompt(ZERO_SHOT_SYSTEM_MESSAGE)
few_shot_prompt = create_prompt(FEW_SHOT_SYSTEM_MESSAGE, few_shot_examples)
cot_few_shot_prompt = create_prompt(COT_FEW_SHOT_SYSTEM_MESSAGE, few_shot_examples)

few_shot_prompt[:4]

In [ ]:
@dataclass
class SimpleResponse:
    content: str


class KeywordSentimentLLM:
    """Fallback local : utile pour tester la pipeline sans API externe."""

    positive_words = {
        "wonderful", "beautiful", "loved", "amazing", "brilliant", "inspiring",
        "good", "great", "excellent", "best", "enjoyed", "perfect",
    }
    negative_words = {
        "boring", "weak", "terrible", "awful", "predictable", "regret",
        "bad", "worst", "slow", "confusing", "poor", "hate",
    }

    def invoke(self, messages):
        review = messages[-1]["content"].lower()
        pos = sum(word in review for word in self.positive_words)
        neg = sum(word in review for word in self.negative_words)
        return SimpleResponse("positive" if pos >= neg else "negative")


llm_local = KeywordSentimentLLM()

# Option OpenAI si OPENAI_API_KEY est configuree dans .env :
# from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
# load_dotenv(override=True)
# llm_openai = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Option Ollama si un modele local est disponible :
# from langchain_ollama import ChatOllama
# llm_ollama = ChatOllama(model="llama3.2", temperature=0)

In [ ]:
def normaliser_prediction(text):
    text = text.strip().lower()
    if "negative" in text:
        return "negative"
    if "positive" in text:
        return "positive"
    return "unknown"


def evaluate_prompt(prompt, gold_examples_json, user_message_template, llm, verbose=False):
    y_true = []
    y_pred = []

    for example in json.loads(gold_examples_json):
        user_input = [
            {
                "role": "user",
                "content": user_message_template.format(movie_review=example["text"]),
            }
        ]

        try:
            response = llm.invoke(prompt + user_input)
            prediction = normaliser_prediction(response.content)
        except Exception as exc:
            print("Erreur LLM :", exc)
            prediction = "unknown"

        y_true.append(example["sentiment"])
        y_pred.append(prediction)

        if verbose:
            print("pred=", prediction, "| true=", example["sentiment"])

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
    }


evaluate_prompt(zero_shot_prompt, gold_examples, USER_MESSAGE_TEMPLATE, llm_local, verbose=True)

In [ ]:
resultats = {
    "zero_shot": evaluate_prompt(zero_shot_prompt, gold_examples, USER_MESSAGE_TEMPLATE, llm_local),
    "few_shot": evaluate_prompt(few_shot_prompt, gold_examples, USER_MESSAGE_TEMPLATE, llm_local),
    "cot_few_shot": evaluate_prompt(cot_few_shot_prompt, gold_examples, USER_MESSAGE_TEMPLATE, llm_local),
}

pd.DataFrame(resultats).T

In [ ]:
def repeated_evaluation(dataset_examples, gold_dataset, llm, n_runs=10, n_examples=2):
    scores = []
    for run in tqdm(range(n_runs)):
        examples = create_examples(dataset_examples, n=n_examples, random_state=run)
        prompt = create_prompt(FEW_SHOT_SYSTEM_MESSAGE, examples)
        gold = (
            gold_dataset.sample(n=min(20, len(gold_dataset)), random_state=run + 100)
            .to_json(orient="records", force_ascii=False)
        )
        metrics = evaluate_prompt(prompt, gold, USER_MESSAGE_TEMPLATE, llm)
        scores.append(metrics["f1"])

    scores = np.array(scores)
    return {
        "mean_f1": scores.mean(),
        "std_f1": scores.std(),
        "runs": scores,
    }


resume = repeated_evaluation(examples_df, gold_examples_df, llm_local, n_runs=10, n_examples=2)
print(f"F1 moyen : {resume['mean_f1']:.3f}")
print(f"Ecart-type : {resume['std_f1']:.3f}")
print("Scores :", resume["runs"])

## Synthèse

La démarche reproduit celle de la vidéo :

1. Charger et inspecter le dataset IMDb.
2. Ajouter une colonne textuelle `sentiment` (`positive` / `negative`).
3. Séparer les exemples few-shot des gold examples.
4. Construire un prompt zero-shot.
5. Construire un prompt few-shot équilibré en exemples positifs/négatifs.
6. Construire une variante chain-of-thought/few-shot avec instructions supplémentaires.
7. Évaluer les prompts avec F1-score, accuracy, precision et recall.
8. Répéter l’évaluation pour mesurer la stabilité du prompt.